# Training dari GitHub -- OTENTIK (Deteksi Wajah Asli vs AI)

Notebook ini pakai kode langsung dari GitHub repo-mu (bukan menulis ulang
tiap file lewat `%%writefile` seperti notebook sebelumnya) -- lebih
gampang dikembangkan: edit kode di lokal/VS Code, `git push`, lalu tinggal
clone/pull di sini.

**Sebelum pakai notebook ini:** ganti `USERNAME/NAMA-REPO` di cell bawah
dengan alamat repo GitHub-mu sendiri (lihat `docs/GITHUB_SETUP.md` di
repo untuk cara upload pertama kali).

---
### Langkah 0 -- Aktifkan GPU
**Runtime > Change runtime type > Hardware accelerator > GPU (T4)** > Save.


In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("✅ GPU terdeteksi:", gpus) if gpus else print("⚠️ Tidak ada GPU, cek Runtime > Change runtime type.")


---
### Langkah 1 -- Clone repo dari GitHub

In [ ]:
# GANTI dengan alamat repo-mu sendiri
REPO_URL = "https://github.com/USERNAME/NAMA-REPO.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

!git clone {REPO_URL}
%cd {REPO_NAME}
!ls


---
### Langkah 2 -- Install dependency

In [ ]:
!pip install -q kaggle scikit-learn pillow matplotlib
print("Selesai.")


---
### Langkah 3 -- Upload model lama (opsional, untuk warm-start/transfer learning)

Kalau mau lanjut training dari model yang sudah ada (bukan dari nol),
upload file `.keras`-nya di sini. Kalau tidak, lewati cell ini saja --
`src/train.py` otomatis training dari nol kalau filenya tidak ditemukan.


In [ ]:
from google.colab import files
import os

print("Upload model lama (.keras) kalau ada, atau Cancel untuk skip:")
try:
    uploaded = files.upload()
    os.makedirs("models", exist_ok=True)
    for fname in uploaded:
        target = "models/cifake_cnn_best.keras"
        os.rename(fname, target)
        print(f"Disimpan sebagai {target}")
except Exception as e:
    print("Dilewati / tidak ada file diupload.")


---
### Langkah 4 -- Download dataset dari Kaggle

1. Buka https://www.kaggle.com/settings/account -> bagian **API** -> **Create New Token**.
2. Jalankan cell di bawah, upload `kaggle.json`.


In [ ]:
from google.colab import files
print("Upload kaggle.json:")
uploaded = files.upload()

import os
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(list(uploaded.values())[0])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("kaggle.json terpasang.")


In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data --unzip
print("Download & extract selesai.")


In [ ]:
# Rapikan struktur folder kalau perlu (otomatis, aman dijalankan meski sudah pas)
import os, shutil, glob

def find_and_flatten(split, cls):
    target = f"data/{split}/{cls}"
    if os.path.isdir(target) and len(os.listdir(target)) > 0:
        return
    candidates = [c for c in glob.glob(f"data/**/{cls}", recursive=True)
                  if split.lower() in c.lower() and os.path.isdir(c)]
    if candidates:
        os.makedirs(f"data/{split}", exist_ok=True)
        if os.path.abspath(candidates[0]) != os.path.abspath(target):
            shutil.move(candidates[0], target)
            print(f"Dipindahkan: {candidates[0]} -> {target}")

for split in ["train", "valid", "test"]:
    for cls in ["real", "fake"]:
        find_and_flatten(split, cls)

print("\nStruktur akhir:")
!find data -maxdepth 2 -type d | sort


---
### Langkah 5 -- Training

In [ ]:
!python -m src.train


In [ ]:
from IPython.display import Image, display
display(Image("reports/training_history.png"))


---
### Langkah 6 -- Evaluasi

In [ ]:
!python -m src.evaluate


In [ ]:
from IPython.display import Image, display
display(Image("reports/confusion_matrix.png"))
with open("reports/classification_report.txt") as f:
    print(f.read())


---
### Langkah 7 -- Export ke TensorFlow Lite

In [ ]:
!python -m src.export_tflite


---
### Langkah 8 -- Simpan hasil

**Opsi A -- download ke laptop:**


In [ ]:
import shutil
shutil.make_archive("hasil_training", "zip", ".", "models")
shutil.make_archive("hasil_laporan", "zip", ".", "reports")

from google.colab import files
files.download("hasil_training.zip")
files.download("hasil_laporan.zip")


**Opsi B -- push balik ke GitHub** (kalau mau simpan model/hasil jadi
bagian repo, misal lewat GitHub Releases manual, atau commit kode yang
diubah di Colab):

```python
!git config --global user.email "emailmu@example.com"
!git config --global user.name "Nama Kamu"
!git add -A
!git commit -m "update hasil training dari Colab"
# ganti <TOKEN> dengan Personal Access Token GitHub-mu
!git push https://<TOKEN>@github.com/USERNAME/NAMA-REPO.git main
```

---
### Catatan
- Model hasil training TIDAK otomatis ter-commit ke git (lihat
  `.gitignore` di repo) -- ini sengaja, karena file model besar & sering
  berubah. Simpan lewat GitHub Releases atau download manual seperti di
  atas.
- Lihat `docs/PROJECT_SUMMARY.md` di repo untuk detail lengkap arsitektur,
  jurnal referensi, dan temuan-temuan sepanjang proyek ini.
